In [1]:
!pip install segmentation-models-pytorch
!pip install captum --quiet
!pip install shap lime --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.3/121.3 kB 9.1 MB/s eta 0:00:00
  Created wheel for efficientnet-pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16425 sha256=23a997341fc6bbe3e74bd8004f0dabfae963a3775fa9a80458f617133c0f0fba
  Stored in directory: /root/.cache/pip/wheels/03/3f/e9/911b1bc46869644912bda90a56bcf7b960f20b5187feea3baf
  Created wheel for pretrainedmodels: filename=pretrainedmodels-0.7.4-py3-none-any.whl size=60944 sha256=7b64f543a8b6c8f153c7240e4e869aad7c1939c85c295930e1652630ef5fda96
  Stored in directory: /root/.cache/pip/wheels/35/cb/a5/8f534c60142835bfc889f9a482e4a67e0b817032d9c6883b64
Successfully built efficientnet-pytorch pretrainedmodels
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.1 MB/s eta 0:00:0000:01


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from segmentation_models_pytorch import UnetPlusPlus
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dataset_path = '/kaggle/input/kermany2018/OCT2017 '
train_dir = os.path.join(dataset_path, 'train')
val_dir = os.path.join(dataset_path, 'val')
test_dir = os.path.join(dataset_path, 'test')

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset_full = datasets.ImageFolder(root=train_dir, transform=image_transform)
val_dataset_from_val_folder = datasets.ImageFolder(root=val_dir, transform=image_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=image_transform)

train_indices, additional_val_indices = train_test_split(
    np.arange(len(train_dataset_full)),
    test_size=0.2,
    stratify=train_dataset_full.targets,
    random_state=42
)

train_dataset = Subset(train_dataset_full, train_indices)
additional_val_dataset = Subset(train_dataset_full, additional_val_indices)

val_dataset = additional_val_dataset
if len(val_dataset_from_val_folder) > 0:
    val_dataset = val_dataset + val_dataset_from_val_folder

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train size: {len(train_loader.dataset)}, Val size: {len(val_loader.dataset)}, Test size: {len(test_loader.dataset)}")

class CustomSwinTransformer(nn.Module):
    def __init__(self, num_classes):
        super(CustomSwinTransformer, self).__init__()
        
        # Custom patch embedding
        self.patch_embed = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=4, stride=4, padding=0), # 4x4 embedding size
            nn.BatchNorm2d(96),
            nn.ReLU()
        )

        # Transformer blocks
        self.transformer_blocks = nn.Sequential(
            nn.TransformerEncoderLayer(d_model=96, nhead=4, dim_feedforward=384, activation="gelu"),
            nn.TransformerEncoderLayer(d_model=96, nhead=4, dim_feedforward=384, activation="gelu")
        )

        # Bottleneck layer
        self.bottleneck = nn.Sequential(
            nn.Linear(96 * 56 * 56, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Classification head
        self.classification_head = nn.Linear(256, num_classes)

    def forward(self, x):
        # Patch embedding
        x = self.patch_embed(x)  # Output shape: [B, 96, 56, 56]
        x = x.flatten(2).permute(2, 0, 1)  # Reshape to [56*56, B, 96] for transformer

        # Transformer blocks
        x = self.transformer_blocks(x)  # Shape: [56*56, B, 96]
        x = x.permute(1, 0, 2).flatten(1)  # Reshape to [B, 96 * 56 * 56]

        # Bottleneck and classification
        x = self.bottleneck(x)
        x = self.classification_head(x)
        return x

# Hybrid model combining Unet++ and Custom Swin Transformer
class HybridModel(nn.Module):
    def __init__(self, num_classes):
        super(HybridModel, self).__init__()
        self.segmentation_model = UnetPlusPlus(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=3,
            classes=1
        )
        self.classification_model = CustomSwinTransformer(num_classes=num_classes)

    def forward(self, x):
        seg_map = self.segmentation_model(x)
        seg_map = torch.sigmoid(seg_map)

        seg_map_resized = nn.functional.interpolate(seg_map, size=(224, 224), mode="bilinear", align_corners=False)

        combined = x * seg_map_resized
        return self.classification_model(combined)

# Model, criterion, and optimizer
num_classes = len(train_dataset_full.classes)
model = HybridModel(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# Training loop
def train_model(model, train_loader, val_loader, num_epochs, criterion, optimizer):
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    for epoch in range(num_epochs):
        model.train()
        train_loss, correct_train, total_train = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            correct_train += predicted.eq(labels).sum().item()
            total_train += labels.size(0)

        train_acc = correct_train / total_train
        train_losses.append(train_loss / len(train_loader))
        train_accuracies.append(train_acc)

        model.eval()
        val_loss, correct_val, total_val = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                correct_val += predicted.eq(labels).sum().item()
                total_val += labels.size(0)

        val_acc = correct_val / total_val
        val_losses.append(val_loss / len(val_loader))
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch+1}/{num_epochs}], "
              f"Train Loss: {train_losses[-1]:.4f}, Train Acc: {train_accuracies[-1]:.4f}, "
              f"Val Loss: {val_losses[-1]:.4f}, Val Acc: {val_accuracies[-1]:.4f}")

    return train_losses, val_losses, train_accuracies, val_accuracies

# Training the model
num_epochs = 30
train_losses, val_losses, train_accuracies, val_accuracies = train_model(
    model, train_loader, val_loader, num_epochs, criterion, optimizer
)

Using device: cuda
Train size: 66787, Val size: 16729, Test size: 968


Downloading: "https://download.pytorch.org/models/resnet34-333f7ec4.pth" to /root/.cache/torch/hub/checkpoints/resnet34-333f7ec4.pth
100%|██████████| 83.3M/83.3M [00:00<00:00, 497MB/s]


In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import seaborn as sns
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Visualization of Segmentation Results
def visualize_segmentation(model, loader, num_samples=5):
    model.eval()
    samples_shown = 0

    for images, _ in loader:
        images = images.to(device)

        with torch.no_grad():
            seg_maps = model.segmentation_model(images)
            seg_maps = torch.sigmoid(seg_maps)

        for i in range(images.size(0)):
            if samples_shown >= num_samples:
                return

            original_image = images[i].cpu().permute(1, 2, 0).numpy()
            original_image = (original_image - original_image.min()) / (original_image.max() - original_image.min())
            segmentation_map = seg_maps[i, 0].cpu().numpy()
            segmentation_map = (segmentation_map - segmentation_map.min()) / (segmentation_map.max() - segmentation_map.min())

            overlay = original_image.copy()
            overlay[:, :, 0] += segmentation_map

            plt.figure(figsize=(15, 5))
            plt.subplot(1, 3, 1)
            plt.imshow(original_image)
            plt.title("Original Image")
            plt.axis("off")

            plt.subplot(1, 3, 2)
            plt.imshow(segmentation_map, cmap="jet")
            plt.title("Segmentation Map")
            plt.axis("off")

            plt.subplot(1, 3, 3)
            plt.imshow(overlay)
            plt.title("Overlay")
            plt.axis("off")

            plt.tight_layout()
            plt.show()
            samples_shown += 1

# Classification Report and Confusion Matrix for Test Data
def plot_classification_report_and_confusion_matrix(model, loader, class_names, dataset_name="Test Data"):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    print(f"\nClassification Report ({dataset_name}):\n", 
          classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_percent, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted Labels")
    plt.ylabel("True Labels")
    plt.title(f"Confusion Matrix with Percentages ({dataset_name})")
    plt.show()

# Loss and Accuracy Plots
def plot_loss_and_accuracy(train_losses, val_losses, train_accuracies, val_accuracies):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss', color='tab:blue')
    plt.plot(val_losses, label='Validation Loss', color='tab:orange')
    plt.legend(loc="upper right")
    plt.title('Loss Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')

    plt.subplot(1, 2, 2)
    plt.plot(train_accuracies, label='Train Accuracy', color='tab:blue')
    plt.plot(val_accuracies, label='Validation Accuracy', color='tab:orange')
    plt.legend(loc="upper left")
    plt.title('Accuracy Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.tight_layout()
    plt.show()

# ROC Curve
def plot_roc_curve(model, loader, class_names):
    model.eval()
    y_true, y_score = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            y_true.extend(labels.cpu().numpy())
            y_score.extend(probabilities.cpu().numpy())

    y_true = label_binarize(y_true, classes=list(range(len(class_names))))
    y_score = np.array(y_score)

    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(len(class_names)):
        fpr[i], tpr[i], _ = roc_curve(y_true[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    plt.figure(figsize=(10, 8))
    for i in range(len(class_names)):
        plt.plot(fpr[i], tpr[i], label=f'{class_names[i]} (AUC = {roc_auc[i]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve for Multi-Class Classification')
    plt.legend(loc="lower right")
    plt.show()

# Final Performance Metrics
def generate_final_report(model, loader, class_names):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")

    return accuracy, precision, recall, f1

# Generate Tables and Plots
class_names = train_dataset_full.classes
visualize_segmentation(model, test_loader, num_samples=5)
plot_classification_report_and_confusion_matrix(model, test_loader, class_names, dataset_name="Test Data")
plot_classification_report_and_confusion_matrix(model, val_loader, class_names, dataset_name="Validation Data")
plot_loss_and_accuracy(train_losses, val_losses, train_accuracies, val_accuracies)
plot_roc_curve(model, test_loader, class_names)

accuracy, precision, recall, f1 = generate_final_report(model, test_loader, class_names)
final_table = {
    "Model": ["Custom Hybrid Model"],
    "Accuracy": [f"{accuracy:.4f}"],
    "Precision": [f"{precision:.4f}"],
    "Recall": [f"{recall:.4f}"],
    "F1-Score": [f"{f1:.4f}"]
}

df = pd.DataFrame(final_table)
tools.display_dataframe_to_user(name="Model Performance Metrics", dataframe=df)

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

def visualize_segmentation(model, loader, num_samples=5):
    """
    Visualize the original images, segmentation maps, and their overlays.
    
    Args:
        model: The trained segmentation model.
        loader: DataLoader for the images.
        num_samples: Number of samples to visualize.
    """
    model.eval()
    samples_shown = 0

    for images, _ in loader:
        images = images.to(device)

        with torch.no_grad():
            seg_maps = model.segmentation_model(images)
            seg_maps = torch.sigmoid(seg_maps)

        for i in range(images.size(0)):
            if samples_shown >= num_samples:
                return

            original_image = images[i].cpu().permute(1, 2, 0).numpy()
            original_image = (original_image - original_image.min()) / (original_image.max() - original_image.min())
            segmentation_map = seg_maps[i, 0].cpu().numpy()
            segmentation_map = (segmentation_map - segmentation_map.min()) / (segmentation_map.max() - segmentation_map.min())

            overlay = original_image.copy()
            overlay[:, :, 0] += segmentation_map

            plt.figure(figsize=(15, 5))
            plt.subplot(1, 3, 1)
            plt.imshow(original_image)
            plt.title("Original Image")
            plt.axis("off")

            plt.subplot(1, 3, 2)
            plt.imshow(segmentation_map, cmap="jet")
            plt.title("Segmentation Map")
            plt.axis("off")

            plt.subplot(1, 3, 3)
            plt.imshow(overlay)
            plt.title("Overlay")
            plt.axis("off")

            plt.tight_layout()
            plt.show()
            samples_shown += 1

visualize_segmentation(model, test_loader, num_samples=5)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2

class GradCAM:
    def __init__(self, model, target_layer):
        """
        Initialize the Grad-CAM object.
        
        Args:
            model: The trained model.
            target_layer: The specific layer to compute Grad-CAM on.
        """
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_registered = False
        self.register_hooks()

    def register_hooks(self):
        """
        Register hooks to capture gradients and activations.
        """
        def forward_hook(module, input, output):
            self.activations = output

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0]

        for name, module in self.model.named_modules():
            if name == self.target_layer:
                module.register_forward_hook(forward_hook)
                module.register_backward_hook(backward_hook)
                self.hook_registered = True
                break

        if not self.hook_registered:
            raise ValueError(f"Layer '{self.target_layer}' not found in the model.")

    def generate_cam(self, inputs, target_class=None):
        """
        Generate the Grad-CAM heatmap for a specific class.
        
        Args:
            inputs: Input image tensor.
            target_class: Class index for which Grad-CAM is generated. If None, the predicted class is used.
            
        Returns:
            heatmap: Grad-CAM heatmap (numpy array).
        """
        self.model.eval()
        inputs = inputs.to(device).unsqueeze(0)

        outputs = self.model(inputs)
        if target_class is None:
            target_class = outputs.argmax(dim=1).item()

        loss = outputs[:, target_class]
        self.model.zero_grad()
        loss.backward(retain_graph=True)

        gradients = self.gradients.cpu().numpy()[0]
        activations = self.activations.cpu().numpy()[0]

        # Compute Grad-CAM
        weights = np.mean(gradients, axis=(1, 2))
        cam = np.zeros(activations.shape[1:], dtype=np.float32)

        for i, w in enumerate(weights):
            cam += w * activations[i]

        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (inputs.shape[3], inputs.shape[2]))
        cam = (cam - cam.min()) / (cam.max() - cam.min())  # Normalize between 0 and 1
        return cam

def visualize_gradcam(grad_cam, loader, num_samples=5, target_class=None):
    """
    Visualize Grad-CAM heatmap with original images.
    
    Args:
        grad_cam: GradCAM instance.
        loader: DataLoader for images.
        num_samples: Number of samples to visualize.
        target_class: Class index to visualize. If None, use predicted class.
    """
    samples_shown = 0

    for images, _ in loader:
        images = images.to(device)

        for i in range(images.size(0)):
            if samples_shown >= num_samples:
                return

            image = images[i].cpu().permute(1, 2, 0).numpy()
            image = (image - image.min()) / (image.max() - image.min())

            heatmap = grad_cam.generate_cam(images[i], target_class)
            heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
            heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
            overlay = cv2.addWeighted(image, 0.6, heatmap_colored / 255.0, 0.4, 0)

            plt.figure(figsize=(15, 5))
            plt.subplot(1, 3, 1)
            plt.imshow(image)
            plt.title("Original Image")
            plt.axis("off")

            plt.subplot(1, 3, 2)
            plt.imshow(heatmap_colored)
            plt.title("Grad-CAM Heatmap")
            plt.axis("off")

            plt.subplot(1, 3, 3)
            plt.imshow(overlay)
            plt.title("Overlay")
            plt.axis("off")

            plt.tight_layout()
            plt.show()
            samples_shown += 1

# Specify the target layer for Grad-CAM
target_layer = "layer4"  # Change this to the desired layer name
grad_cam = GradCAM(model, target_layer)

# Visualize Grad-CAM for test loader
visualize_gradcam(grad_cam, test_loader, num_samples=5)

In [ ]:
# =====================
# 4. Visualization
# =====================
# Loss and accuracy curves
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.title("Loss Over Epochs")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Validation Accuracy")
plt.title("Accuracy Over Epochs")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# =====================
# 5. Evaluation Metrics
# =====================
# Example classification report and confusion matrix
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for images, _, labels in val_loader:
        images = images.to(device)
        _, class_output = model(images)
        _, predicted = class_output.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=["Class 0", "Class 1", "Class 2"]))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Class 0", "Class 1", "Class 2", "Class 3"], yticklabels=["Class 0", "Class 1", "Class 2", "Class 3"])
plt.title("Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.show()